In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
# Load the Truth file
expansion_truth = pd.read_csv('HeroExpansion.csv')

# Create a lookup set for the "Perfect Match" check
# We keep the casing as it appears in your HeroExpansion.csv
official_names = set(expansion_truth['Hero Name'].str.strip().tolist())

In [3]:
def normalize_hero(name):
    # --- FAILSAFE ---
    if pd.isna(name) or str(name).lower() in ['nan', '', '-', '0', '0.0', 'none']:
        return ""
    
    n = str(name).strip()

    # --- SECTION 1: THE 100% CONFORMANCE CHECK ---
    if n in official_names:
        return n

    # --- SECTION 2: DIRECT EXACT-MATCH REPAIR ---
    # Put all your standard variants here. This guarantees they fix instantly 
    # and safely without breaking other hero names.
    exact_mappings = {
        'Bruce Banner': 'Bruce Banner (World War Hulk)',
        'Black Widow': 'Black Widow (Avengers)',
        'Black Panther': 'Black Panther (Illuminati)',
        'Black Panther (Avengers)': 'Black Panther (MCU)',
        'Peter Parker': 'Peter Parker (Civil War)',
        'Peter Parker/Aunt May': 'Peter Parker (Civil War)',
        'Peter Parker, Homecoming': 'Peter Parker (Homecoming)',
        'Dr. Strange': 'Dr. Strange (Illuminati)',
        'Doctor Strange (Doctor Strange and the Shadows of Nightmare)': 'Doctor Strange (Avengers)',
        'Scott Lang, Cat Burgler': 'Scott Lang, Cat Burglar',
        'Super Skrull': 'Super-Skrull',
        'Elsa Bloodstone': 'Elsa Bloodstone (Marvel Knights)',
        'Captain Marvel': 'Captain Marvel (Secret Wars Vol. 1)',
        'Ant-Man': 'Ant-Man (Ant-Man)',
        'Wasp': 'Wasp (Ant-Man)',
        'Drax': 'Drax the Destroyer',
        'Daredevil (Iron Fist)': 'Daredevil (Avengers)',
        'Steve Rogers, Director of Shield': 'Steve Rogers, Director of SHIELD'
    }

    if n in exact_mappings:
        n = exact_mappings[n]
        if n in official_names:
            return n

    # --- SECTION 3: GLOBAL FORMATTING & CLEANUP ---
    # Case-insensitive Spider-Man fix
    n = re.sub(r'Spider\s*man', 'Spider-Man', n, flags=re.IGNORECASE)
    
    # Text cleanups
    n = n.replace(' The ', ' the ')
    n = n.replace(' & ', ' and ')
    n = n.replace('.', '')  # Strip dots

    if n in official_names: 
        return n

    # Handle "The" prefix to suffix (e.g. "The Punisher" -> "Punisher, The")
    if n.startswith('The '):
        n = n[4:] + ", The"
        if n in official_names: 
            return n

    # --- FINAL RE-CHECK TRUTH ---
    if n in official_names:
        return n
            
    print(f"False Name: {name} -> Evaluated As: {n}")
    return n

In [4]:
def parse_bracketed_data(df, col_idx, prefix):
    current_cols = list(df.columns)
    col_name = current_cols[col_idx]
    dropped = []

    # A. Combine text if the next column is an 'Unnamed' neighbor
    main_text = df[col_name].astype(str).str.strip()
    if col_idx + 1 < len(current_cols) and "Unnamed" in current_cols[col_idx + 1]:
        neighbor_name = current_cols[col_idx + 1]
        neighbor_text = df[neighbor_name].astype(str).str.strip()
        parse_text = main_text.where(main_text.str.contains(r'\(', na=False), 
                                     main_text + " " + neighbor_text)
        dropped.append(neighbor_name)
    else:
        parse_text = main_text

    # B. Extract pieces
    # Remove brackets AND remove the rank suffix
    temp_val = parse_text.str.replace(r'\(.*?\)', '', regex=True)
    df[f"{prefix}_Value"] = temp_val.str.replace(r'\d+(?:st|nd|rd|th)', '', regex=True).str.strip()
    
    df[f"{prefix}_Points"] = parse_text.str.extract(r'\((\d*\.?\d+)\)', expand=False)
    df[f"{prefix}_Rank"] = parse_text.str.extract(r'(\d+(?:st|nd|rd|th))', expand=False)

    dropped.append(col_name)
    return dropped

In [5]:
def dynamically_rename_columns(df, official_names):
    hero_count = 1
    rename_map = {}
    cols_to_drop = []
    
    # 1. Standardize headers
    df.columns = [re.sub(r'\s+', ' ', str(col)).strip() for col in df.columns]
    
    current_cols = list(df.columns)
    for i, col in enumerate(current_cols):
        # Already processed this as a neighbor? Skip it.
        if col in cols_to_drop: continue

        # Clean data for consistent regex matching
        df[col] = df[col].astype(str).str.replace(r'[\n\t\r]', ' ', regex=True).str.strip()
        col_clean = col.lower()
        
        # 2. CALL HELPER FOR TRIPLE-THREAT COLUMNS
        if "escape" in col_clean:
            cols_to_drop.extend(parse_bracketed_data(df, i, "Escapes"))
            continue
            
        if "per turn" in col_clean or "ppt" in col_clean:
            cols_to_drop.extend(parse_bracketed_data(df, i, "PPT"))
            continue
            
        if "turn" in col_clean:
            cols_to_drop.extend(parse_bracketed_data(df, i, "Turns"))
            continue

        # 3. WIN/POINTS LOGIC
        if df[col].str.contains(r'(?:Yes|No)\s*\(', na=False).any():
            df['Win_Value'] = df[col].str.extract(r'(Yes|No)', expand=False)
            df['Win_Points'] = pd.to_numeric(df[col].str.extract(r'\((\d*?\.?\d+)\)', expand=False), errors='coerce')
            cols_to_drop.append(col)
            continue 

        # 4. HERO IDENTIFICATION
        if df[col].isin(official_names).any():
            rename_map[col] = f"Hero {hero_count}"
            hero_count += 1


    # 5. Finalize Drops (Add Comments and Setup here)
    for target in ['Comments', 'Setup']:
        if target in df.columns:
            cols_to_drop.append(target)

    df.rename(columns=rename_map, inplace=True)
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore', inplace=True)
    

    # 6. CONSOLIDATE OTHER CLUSTERS
    clusters = {
        'VP_Diff': ['VP Difference', 'VP Diff.'],
        'Fun_Level': ['Fun Level', 'Fun Level (no slash)', 'Fun Lv.'],
        'Name_Clean': ['Name', 'Name(s)'],
        'VP_Pile_1': ['Victory Pile 1', 'Pile 1', 'VP 1'],
        'VP_Pile_2': ['Victory Pile 2', 'Pile 2', 'VP 2'],
        'Total_VP': ['Total VP', 'VP'],
        'Bystanders': [c for c in df.columns if 'stander' in c.lower() or 'bst.' in c.lower()],
        'Attempt': ['Attempt Submitted', 'Attempt Used', 'Used']
    }

    for master, aliases in clusters.items():
        existing = [c for c in aliases if c in df.columns]
        if not existing: continue
        
        master_col = sorted(existing, key=lambda x: df[x].count(), reverse=True)[0]
        df.rename(columns={master_col: master}, inplace=True)
        
        for extra in [e for e in existing if e != master_col]:
            df[master] = df[master].replace(['nan', 'None', ''], np.nan).fillna(df[extra].replace(['nan', 'None', ''], np.nan))
            df.drop(columns=[extra], inplace=True)

    
            
    return df

In [6]:
def impute_victory_points(df):
    vp_cols = ['VP_Pile_1', 'VP_Pile_2', 'VP_Diff','Total_VP']
    
    # 1. Ensure columns exist. If VP_Diff is missing, create it as NaN
    if 'VP_Diff' not in df.columns:
        df['VP_Diff'] = np.nan
        
    for col in vp_cols:
        if col in df.columns:
            # Force to numeric (NaNs become actual np.nan objects)
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 2. Recalculate VP_Diff BEFORE filling with 0
    if 'VP_Pile_1' in df.columns and 'VP_Pile_2' in df.columns:
        # Now this won't KeyError because we ensured it exists in Step 1
        mask = (df['VP_Diff'].isna()) | (df['VP_Diff'] == 0)
        df.loc[mask, 'VP_Diff'] = (df['VP_Pile_1'] - df['VP_Pile_2']).abs()

    # 3. Final Sweep: Fill remaining NaNs with 0 and force INT
    for col in vp_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)
    if 'Total_VP' in df.columns:
        v1 = pd.to_numeric(df.get('VP_Pile_1', 0), errors='coerce').fillna(0)
        v2 = pd.to_numeric(df.get('VP_Pile_2', 0), errors='coerce').fillna(0)
        df['Total_VP'] = df['Total_VP'].replace(['nan', 'None', ''], np.nan).fillna(v1 + v2)

    
            
    return df

In [7]:
def hero_picks(df, official_names):
    # 1. SETUP: Ensure all Hero columns are normalized strings right away!
    hero_cols = [c for c in df.columns if 'Hero' in c]
    for col in hero_cols:
        # Cast to string, strip whitespace, and apply your normalization rules
        df[col] = df[col].astype(str).str.strip().apply(normalize_hero)

    # 2. DEFAULT: Start by assuming Hero 1 and Hero 2 were the picks
    df['Pick 1'] = df.get('Hero 1', np.nan)
    df['Pick 2'] = df.get('Hero 2', np.nan)

    # 3. ATTEMPT & H3/H4 LOGIC
    if 'Attempt' in df.columns and 'Hero 3' in df.columns and 'Hero 4' in df.columns:
        
        # Now this works! Hero 3 and 4 have already been normalized to match official_names
        h3_valid = df['Hero 3'].isin(official_names)
        h4_valid = df['Hero 4'].isin(official_names)
        
        # Identify "Second Attempt" rows
        is_second = df['Attempt'].astype(str).str.lower().str.contains('second', na=False)
        
        # SWITCH CONDITION: 
        switch_mask = is_second & h3_valid & h4_valid
        
        df.loc[switch_mask, 'Pick 1'] = df['Hero 3']
        df.loc[switch_mask, 'Pick 2'] = df['Hero 4']

    sp22_data = df.loc[df['Unique_Pod_ID'].astype(str).str.contains('Sp22', na=False), 'Unique_Pod_ID']
    if not sp22_data.empty:
        print("Assume Hero 1 2")
        print(sp22_data.to_string())
    
    # 4. DOUBLE CHECK: Verification Step
    df.loc[~df['Pick 1'].isin(official_names), 'Pick 1'] = np.nan
    df.loc[~df['Pick 2'].isin(official_names), 'Pick 2'] = np.nan

    sp22_data = df.loc[df['Unique_Pod_ID'].astype(str).str.contains('Sp22', na=False), 'Unique_Pod_ID']
    if not sp22_data.empty:
        print("Checks Hero name")
        print(sp22_data.to_string())

    # 5. FINAL CLEANUP: Drop any column with the word "Hero"
    cols_to_drop = [c for c in df.columns if 'Hero' in c and c not in ['Pick 1', 'Pick 2']]
    df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

    sp22_check = df[df['Unique_Pod_ID'].astype(str).str.contains('Sp22', na=False)]
    if not sp22_check.empty:
        print("\n=== DEBUG: HERO PICKS RIGHT BEFORE DROPNA ===")
        print(sp22_check[['Unique_Pod_ID', 'Name_Clean', 'Pick 1', 'Pick 2']].to_string() if 'Name_Clean' in sp22_check.columns else sp22_check[['Unique_Pod_ID', 'Pick 1', 'Pick 2']].to_string())

    # Remove rows where we couldn't find a valid Hero Pick
    df = df.dropna(subset=['Pick 1', 'Pick 2'])

    sp22_check = df[df['Unique_Pod_ID'].astype(str).str.contains('Sp22', na=False)]
    if not sp22_check.empty:
        print("\n=== DEBUG: HERO PICKS AFTER DROPNA ===")
        print(sp22_check[['Unique_Pod_ID', 'Name_Clean', 'Pick 1', 'Pick 2']].to_string() if 'Name_Clean' in sp22_check.columns else sp22_check[['Unique_Pod_ID', 'Pick 1', 'Pick 2']].to_string())

    return df

In [8]:
def clean_legndary_data(file_path):
    # 1. LOAD AND INITIAL CLEANING
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    df.columns = df.columns.str.strip() 

    # Early purge of obvious header/empty rows
    if 'Name(s)' in df.columns:
        df = df[~df['Name(s)'].isin(['Rank', 'Points', 'Name', 'Name(s)', 'nan', ''])]
    df = df.dropna(how='all')

    # 2. DYNAMIC COLUMN GAP DROPPER (Villain Deck Logic)
    if 'Cards Left in Villain Deck' in df.columns:
        start_idx = df.columns.get_loc('Cards Left in Villain Deck')
        # We only want to drop the specific 'Cards Left' column and any UNNAMED gaps following it
        gap_cols = [df.columns[start_idx]] # Add the villain deck column itself
        for col in df.columns[start_idx + 1:]:
            if 'Unnamed' in col:
                gap_cols.append(col)
            else:
                break # Stop as soon as we hit a real name like "Escaped Villains"
        df.drop(columns=gap_cols, inplace=True)
            
    # De-duplicate names to prevent column collisions
    new_cols = []
    for col in df.columns:
        count = new_cols.count(col)
        new_cols.append(f"{col}.{count}" if count > 0 else col)
    df.columns = new_cols

    # 3. GENERATE TOURNEY ID
    raw_filename = os.path.basename(file_path).replace(".csv", "")
    clean_name = re.sub(r'W\d+|Week\s*\d+|Results|Copy\s*of\s*', '', raw_filename, flags=re.IGNORECASE)
    tourney_id = clean_name.replace('-', '').replace('  ', ' ').strip().replace(' ', '')
    
    # 4. DYNAMIC RENAMING & EXTRACTION (The Magnet + Regex)
    # This now handles: Hero Picks, Turns, Turn Rank, Win_Value, and Win_Points
    df = dynamically_rename_columns(df, official_names)

    # 5. ELIMINATION POD FIX
    def convert_elim_pods(val):
        val_str = str(val).strip().upper()
        if val_str.startswith('E'):
            try:
                # E-Pods are 101, 102, etc.
                return int(re.search(r'\d+', val_str).group()) + 100
            except:
                return np.nan
        return pd.to_numeric(val_str, errors='coerce')
    
    if 'Pod' in df.columns:
        # Fill the Pod number down into the NaNs created by merged cells
        df['Pod'] = df['Pod'].replace(['-', '0', '0.0', 'nan', 'None'], np.nan)
        df['Pod'] = df['Pod'].ffill()
        df['Pod'] = df['Pod'].apply(convert_elim_pods)

    # 6. CRITICAL: DROP GHOST ROWS & DQ ENTRIES
    # We use 'Win_Value' and 'Win_Points' as the "Anchor of Truth"
    if 'Win_Value' in df.columns:
        # Drop if Win_Value is empty OR if the extraction in #4 failed to find points
        df = df[
            df['Win_Value'].notna() & 
            ~df['Win_Value'].astype(str).str.lower().isin(['nan', 'none', '-', '']) &
            df['Win_Points'].notna()
        ]

    # Ensure Pod is a clean integer after the drop
    if 'Pod' in df.columns:
        df['Pod'] = df['Pod'].fillna(0).astype(int)

    # 7. UNIQUE ID GENERATION
    # Uses the cleaned Tourney Name + the filled Pod number
    df['Unique_Pod_ID'] = tourney_id + "_Pod_" + df['Pod'].astype(str)

    # Final cleanup of strings before returning
    df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)

    
    df = impute_victory_points(df)

    df = hero_picks(df, official_names)
    sp22_data = df.loc[df['Unique_Pod_ID'].astype(str).str.contains('Sp22', na=False), 'Unique_Pod_ID']
    if not sp22_data.empty:
        print("Hero Pick")
        print(sp22_data.to_string())

    if 'Bystanders' in df.columns:
        df['Bystanders'] = df['Bystanders'].replace(['nan', 'None', ''], np.nan).fillna(0)
    if 'Win?' in df.columns:
        df = df[df['Win?'] != '-'].copy()
    
    return df

In [9]:
import glob

# Grabs every CSV file in the "data" folder
file_list = glob.glob("dirty results/*.csv")

all_weeks = []
for file in file_list:
    print(f"Processing: {file}") # This will pinpoint the broken file
    cleaned_df = clean_legndary_data(file)
    all_weeks.append(cleaned_df)

master_df = pd.concat(all_weeks, ignore_index=True)

# Keep columns only if they have 200 or more non-null values
master_df = master_df.dropna(axis=1, thresh=200)

Processing: dirty results\F22W1.csv


C:\Users\beers\AppData\Local\Temp\ipykernel_30712\3773802504.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)


Processing: dirty results\F22W2.csv


C:\Users\beers\AppData\Local\Temp\ipykernel_30712\3773802504.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)


Processing: dirty results\F22W3.csv


...

Processing: dirty results\Su22W6.csv


C:\Users\beers\AppData\Local\Temp\ipykernel_30712\3773802504.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)
C:\Users\beers\AppData\Local\Temp\ipykernel_30712\3773802504.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)


Processing: dirty results\Su22W7.csv
Processing: dirty results\Su22W8.csv


C:\Users\beers\AppData\Local\Temp\ipykernel_30712\3773802504.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)
C:\Users\beers\AppData\Local\Temp\ipykernel_30712\3773802504.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)


Processing: dirty results\Su22W9.csv


In [10]:
master_df['Unique_Pod_ID'].unique()

array(['F22_Pod_1', 'F22_Pod_2', 'F22_Pod_3', 'F22_Pod_4', 'F22_Pod_5',
       'F22_Pod_6', 'F22_Pod_7', 'F22_Pod_8', 'F22_Pod_9', 'F22_Pod_10',
       'F22_Pod_11', 'F22_Pod_12', 'F22_Pod_13', 'F22_Pod_14',
       'F22_Pod_15', 'F22_Pod_16', 'F22_Pod_17', 'F22_Pod_18',
       'F22_Pod_19', 'F22_Pod_20', 'F22_Pod_21', 'F22_Pod_22',
       'F22_Pod_101', 'F22_Pod_23', 'F22_Pod_24', 'F22_Pod_25',
       'F22_Pod_26', 'F22_Pod_102', 'F22_Pod_27', 'F22_Pod_28',
       'F22_Pod_29', 'F22_Pod_103', 'F22_Pod_30', 'F22_Pod_31',
       'F22_Pod_32', 'F22_Pod_104', 'F22_Pod_33', 'F22_Pod_105',
       'F22_Pod_34', 'F22_Pod_107', 'F23_Pod_1', 'F23_Pod_2', 'F23_Pod_3',
       'F23_Pod_4', 'F23_Pod_5', 'F23_Pod_6', 'F23_Pod_7', 'F23_Pod_8',
       'F23_Pod_36', 'F23_Pod_120', 'F23_Pod_121', 'F23_Pod_122',
       'F23_Pod_9', 'F23_Pod_10', 'F23_Pod_11', 'F23_Pod_12',
       'F23_Pod_13', 'F23_Pod_14', 'F23_Pod_15', 'F23_Pod_16',
       'F23_Pod_17', 'F23_Pod_18', 'F23_Pod_19', 'F23_Pod_20',
       '

In [11]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4375 entries, 0 to 4374
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Pod                4375 non-null   int32  
 1   Rank               4375 non-null   object 
 2   Name_Clean         4375 non-null   object 
 3   VP_Pile_1          4375 non-null   int32  
 4   VP_Pile_2          4375 non-null   int32  
 5   Total_VP           4375 non-null   int32  
 6   Win?               4375 non-null   object 
 7   Total Rank Points  4375 non-null   object 
 8   VP_Diff            4375 non-null   int32  
 9   Bystanders         4375 non-null   object 
 10  Fun_Level          4375 non-null   object 
 11  Win_Value          4375 non-null   object 
 12  Win_Points         4375 non-null   float64
 13  Turns_Value        4375 non-null   object 
 14  Turns_Points       4375 non-null   object 
 15  Turns_Rank         4375 non-null   object 
 16  Escapes_Value      4375 

In [12]:
# Force pandas to show every single column name regardless of length
print(master_df.columns.tolist())


['Pod', 'Rank', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'VP_Diff', 'Bystanders', 'Fun_Level', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 'Escapes_Rank', 'PPT_Value', 'PPT_Points', 'PPT_Rank', 'Unique_Pod_ID', 'Pick 1', 'Pick 2', 'Attempt', 'Points Earned', 'Total Points', 'Cards Undercover']


In [13]:
master_df.to_csv('tourney_results_cleaned.csv', index=False)

In [14]:
# Filter for any ID ending in 'Pod_0'
ghost_pods = master_df[master_df['Unique_Pod_ID'].str.endswith('Sp26_Pod_68', na=False)]

# View everything in those rows
# I'm including Setup and Name_Clean to help you identify which files they came from
pd.set_option('display.max_columns', None) # Ensure we see all columns
print(f"Total Ghost Rows found: {len(ghost_pods)}")
print(ghost_pods.head(20))

Total Ghost Rows found: 8
      Pod Rank                              Name_Clean  VP_Pile_1  VP_Pile_2  \
4072   68  1.0                               Doormatte         53         19   
4073   68  2.0                               Freeclint         64         26   
4074   68  3.0  SharkNate-O & onthegeniusofsamrockwell         41         30   
4075   68  4.0                       NadRao & Sparkles         43         25   
4076   68  5.0                             Jmaster1324         43         35   
4077   68  6.0            thedoctormate0806 & mrrj6549         53         46   
4078   68  7.0                       Astigmatic_Oracle         33         36   
4079   68  8.0                              Lasombra81         14         62   

      Total_VP Win? Total Rank Points  VP_Diff Bystanders Fun_Level Win_Value  \
4072        72  1st                28       34         14         3       Yes   
4073        90  1st                22       38         12         5       Yes   
4074      

In [15]:
dfError=pd.read_csv('dirty results/Sp26W6.csv')

In [16]:
dfError.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 37 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Pod                         15 non-null     float64
 1   Setup                       11 non-null     object 
 2   Rank                        165 non-null    float64
 3   Points Earned               100 non-null    object 
 4   Total Points                100 non-null    float64
 5   Name(s)                     100 non-null    object 
 6   Victory
Pile 1              100 non-null    object 
 7   Victory
Pile 2              100 non-null    object 
 8   Total
VP                    100 non-null    object 
 9   Win?                        101 non-null    object 
 10  Unnamed: 10                 101 non-null    object 
 11  Turns                       101 non-null    object 
 12  Unnamed: 12                 101 non-null    object 
 13  Escaped Villains            101 non

In [17]:
dfError.head()

,Pod,Setup,Rank,Points Earned,Total Points,Name(s),Victory\nPile 1,Victory\nPile 2,Total\nVP,Win?,Unnamed: 10,Turns,Unnamed: 12,Escaped Villains,Unnamed: 14,Points per Turn,Unnamed: 16,Total Rank Points,Unnamed: 18,VP Difference,Bystanders Rescued,Cards Undercover,Cards Left in Villain Deck,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Attempt 1 Hero Choices,Unnamed: 30,Attempt 2 Hero Choices,Unnamed: 32,Attempt Submitted,Fun Level,Comments,Results Of Week:
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MS,ST,V,H,B,O,Total,1,2,1,2,NaN,NaN,NaN,6.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rank,Points,Rank,Points,Rank,Points,Rank,Points,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,59.0,Suffering from Summoning Sixness,1.0,(+9),43.0,WilTheGrey,69,33,102,1st,Yes (5),1st,20 (9),1st,0 (5),1st,5.1 (9),28,NaN,36,19,7,4,6,12,7,0,0,29,Gladiator Hulk,Nova (Avengers),I did not make a second attempt.,I did not make a second attempt.,First attempt,5,"Hiroim was definitely the MVP here, and Gladia...",NaN
3,NaN,NaN,2.0,(+8),43.0,R3DJ0N,75,28,103,1st,Yes (5),3rd,23 (7),1st,0 (5),4th,4.478 (6),23,NaN,47,22,3,4,4,13,5,0,0,26,Nova (Avengers),Venompool,Nova (Avengers),Venompool,First attempt,4,Both games went way better than I expected. 1s...,NaN
4,NaN,NaN,3.0,(+7),44.0,ReoFirestar19,63,60,123,1st,Yes (5),6th,26 (4),1st,0 (5),2nd,4.731 (8),22,NaN,3,24,6,4,5,8,2,1,0,20,Bruce Banner,Ronan the Accuser,I did not make a second attempt.,I did not make a second attempt.,First attempt,5,"Got Hiroim's Rare, and with some Rally Kree Wa...",NaN


In [18]:
# Count rows where Bruce Banner is in Pick 1 OR Pick 2
bruce_count = master_df[(master_df["Pick 1"] == "Bruce Banner (World War Hulk)") | (master_df["Pick 2"] == "Bruce Banner (World War Hulk)")].shape[0]

print(f"Bruce Banner (World War Hulk) played in {bruce_count} games.")

Bruce Banner (World War Hulk) played in 71 games.


In [19]:
import pandas as pd
import numpy as np

print("============== MASTER DATAFRAME DIAGNOSTIC REPORT ==============")
total_rows = len(master_df)
print(f"Total rows logged in dataframe: {total_rows}\n")

# 1. Systemic check: Are there any rows where the Pick columns are completely blank?
# We check for actual NaN values or empty strings/spaces
empty_picks = master_df[
    master_df["Pick 1"].isna() | 
    (master_df["Pick 1"].astype(str).str.strip() == "") |
    (master_df["Pick 1"].astype(str).str.lower() == "nan")
]
print(f"1. Rows missing a 'Pick 1' hero selection: {len(empty_picks)}")

if len(empty_picks) > 0:
    print("\n--- Sample of Rows Missing Hero Selections ---")
    print(empty_picks[["Unique_Pod_ID", "Pod", "Name_Clean", "Pick 1", "Pick 2"]].head(10).to_string())

print("\n" + "="*60 + "\n")

# 2. Target Search: Specifically pull Pod 68 by treating it as a raw integer
print("============== TARGET SEARCH: POD 68 DIAGNOSTIC ==============")

# Filter exactly for the integer 68 since column type is int32
pod_68_data = master_df[master_df["Pod"] == 68]

print(f"Total rows found where Pod == 68: {len(pod_68_data)}")

if len(pod_68_data) > 0:
    print("\n--- Raw Data Found for Pod 68 ---")
    # We print the vital tracking information to check if it's blank or populated
    print(pod_68_data[["Unique_Pod_ID", "Pod", "Name_Clean", "Pick 1", "Pick 2", "Win_Value", "Total_VP"]].to_string())
else:
    print("❌ SYSTEMIC ALERT: Pod 68 does not exist as an integer in the 'Pod' column.")
    
    # Backup check: Just in case there's an issue, let's see if it's hiding in Unique_Pod_ID
    pod_68_id_search = master_df[master_df["Unique_Pod_ID"].astype(str).str.contains("68", na=False)]
    print(f"Backup search: Rows containing '68' anywhere inside Unique_Pod_ID text: {len(pod_68_id_search)}")
    if len(pod_68_id_search) > 0:
        print(pod_68_id_search[["Unique_Pod_ID", "Pod", "Name_Clean", "Pick 1", "Pick 2"]].head(5).to_string())

============== MASTER DATAFRAME DIAGNOSTIC REPORT ==============
Total rows logged in dataframe: 4375

1. Rows missing a 'Pick 1' hero selection: 0


============== TARGET SEARCH: POD 68 DIAGNOSTIC ==============
Total rows found where Pod == 68: 26

--- Raw Data Found for Pod 68 ---
     Unique_Pod_ID  Pod                              Name_Clean                               Pick 1                               Pick 2 Win_Value  Total_VP
1364    F24_Pod_68   68                            Avataristche  Captain Marvel (Secret Wars Vol. 1)                    Superior Iron Man       Yes       105
1365    F24_Pod_68   68                                  R3DJ0N           Black Panther (Illuminati)  Captain Marvel (Secret Wars Vol. 1)       Yes       118
1366    F24_Pod_68   68               the.melk.man & Snapcity57                  Ultimate Spider-Man                    Superior Iron Man       Yes        96
1367    F24_Pod_68   68                           Adrian Miller                    

In [20]:
# 1. Filter for rows where the Unique_Pod_ID contains 'Sp26' AND matches pods 67, 68, or 69
sp26_neighborhood_filter = (
    master_df["Unique_Pod_ID"].astype(str).str.contains("Sp26", case=False, na=False) & 
    master_df["Pod"].isin([67, 68, 69])
)

# 2. Pull and sort the data
sp26_neighborhood = master_df[sp26_neighborhood_filter].sort_values(by="Pod")

print("============== 🌸 SP26 NEIGHBORHOOD DIAGNOSTIC (67, 68, 69) ==============")
print(f"Total rows found for Sp26 in this range: {len(sp26_neighborhood)}\n")

if len(sp26_neighborhood) > 0:
    print(sp26_neighborhood[["Unique_Pod_ID", "Pod", "Name_Clean", "Pick 1", "Pick 2", "Win_Value"]].to_string())
else:
    print("❌ ALERT: No Sp26 pods numbered 67, 68, or 69 found.")
    print("Let's look at a few raw Sp26 IDs to see exactly how they are formatted:")
    
    # Backup: print the first 10 rows that contain Sp26 to inspect the text formatting
    any_sp26 = master_df[master_df["Unique_Pod_ID"].astype(str).str.contains("Sp26", case=False, na=False)]
    if len(any_sp26) > 0:
        print(any_sp26[["Unique_Pod_ID", "Pod", "Name_Clean"]].head(10).to_string())
    else:
        print("❌ CRITICAL: 'Sp26' text does not exist anywhere in the Unique_Pod_ID column!")

============== 🌸 SP26 NEIGHBORHOOD DIAGNOSTIC (67, 68, 69) ==============
Total rows found for Sp26 in this range: 24

     Unique_Pod_ID  Pod                              Name_Clean                         Pick 1                            Pick 2 Win_Value
4063   Sp26_Pod_67   67                              Zeromus104                      Hulk 2099                            Rictor       Yes
4064   Sp26_Pod_67   67                                 AntDude                       Weapon H         Falcon and Winter Soldier       Yes
4065   Sp26_Pod_67   67                brentmcampbell & ruuhaku           Weapon X (Wolverine)                         Hulk 2099       Yes
4066   Sp26_Pod_67   67                               jetsetrad                      Hulk 2099                          Fantomex       Yes
4067   Sp26_Pod_67   67                            Dr. Parmesan                      Hulk 2099                                 M       Yes
4068   Sp26_Pod_67   67                        

In [21]:
import pandas as pd

# 1. Filter for all rows containing 'Sp26' in their Unique_Pod_ID
sp26_all = master_df[master_df["Unique_Pod_ID"].astype(str).str.contains("Sp26", case=False, na=False)]

# 2. Extract every parsed pod number we actually have right now
actual_pods = set(sp26_all["Pod"].unique())

# 3. Create the expected sequence from 1 to 80
expected_pods = set(range(1, 81))

# 4. Find which expected numbers are missing entirely
missing_pods = sorted(list(expected_pods - actual_pods))

print("================= 🌸 SP26 COMPLETE SECTOR AUDIT =================")
print(f"Total Unique Sp26 Pods found in data: {len(actual_pods)} out of 80 expected.\n")

if len(missing_pods) == 0:
    print("✅ Systemic Check: No numbers are missing between 1 and 80!")
else:
    print(f"❌ MISSED NUMBERS DETECTED ({len(missing_pods)} total):")
    print(missing_pods)

print("\n" + "="*65 + "\n")

# 5. TROUBLESHOOT: Look for any Sp26 rows that failed your text-to-number parsing!
# We look for rows that say 'Sp26' but didn't get grouped into a standard integer properly
print("================= 🔍 HUNTING FOR MALFORMED STRINGS =================")

# Let's see if there are rows where the text contains '68' but the parsed Pod column is NOT 68
broken_string_hunt = master_df[
    master_df["Unique_Pod_ID"].astype(str).str.contains("Sp26", case=False, na=False) &
    master_df["Unique_Pod_ID"].astype(str).str.contains("68") &
    (master_df["Pod"] != 68)
]

if len(broken_string_hunt) > 0:
    print("Found rows where the ID string mentions '68' but the parsed number column broke:")
    print(broken_string_hunt[["Unique_Pod_ID", "Pod", "Name_Clean", "Pick 1"]])
else:
    print("No malformed '68' ID strings found. If 68 is missing from the list above, it means")
    print("the row is physically not in the dataframe at all.")
    
    # Let's print out 5 rows right around where 68 should be in the raw file to see the jump
    print("\n--- Raw sequence gap inspection ---")
    gap_inspection = sp26_all[sp26_all["Pod"].isin([65, 66, 67, 69, 70])].sort_values("Pod")
    print(gap_inspection[["Unique_Pod_ID", "Pod", "Name_Clean"]].drop_duplicates().head(10).to_string())

================= 🌸 SP26 COMPLETE SECTOR AUDIT =================
Total Unique Sp26 Pods found in data: 80 out of 80 expected.

✅ Systemic Check: No numbers are missing between 1 and 80!


================= 🔍 HUNTING FOR MALFORMED STRINGS =================
No malformed '68' ID strings found. If 68 is missing from the list above, it means
the row is physically not in the dataframe at all.

--- Raw sequence gap inspection ---
     Unique_Pod_ID  Pod           Name_Clean
4046   Sp26_Pod_65   65         ElongatedMan
4047   Sp26_Pod_65   65  Loconnole & Klcgolf
4048   Sp26_Pod_65   65            Kb_driven
4049   Sp26_Pod_65   65              jpilger
4050   Sp26_Pod_65   65            iburnicus
4051   Sp26_Pod_65   65               Ocs333
4052   Sp26_Pod_65   65       caseystahl1125
4053   Sp26_Pod_65   65           Joebro6616
4054   Sp26_Pod_65   65               PHoney
4062   Sp26_Pod_66   66              MRTINCE


In [22]:
import pandas as pd
import re

# Make a clean working copy
audit_df = master_df.copy()

# 1. Systemic Fix: Extract the raw numeric pod number from the string to bypass alphabetical sorting
# This regex looks for the numbers immediately following 'Pod_' or at the end of the ID string
def extract_true_pod_number(row):
    # Try to grab it from the Unique_Pod_ID text string first
    id_str = str(row.get("Unique_Pod_ID", ""))
    match = re.search(r'Pod_(\d+)', id_str, re.IGNORECASE)
    if match:
        return int(match.group(1))
    # Fallback to your engineered 'Pod' column if the text search misses
    return int(row["Pod"])

audit_df["True_Numeric_Pod"] = audit_df.apply(extract_true_pod_number, axis=1)

# 2. Systemic Fix: Extract the Season Code so we can group seasons together chronologically
# This looks for prefixes like 'Sp26', 'Fa25', etc.
def extract_season_prefix(row):
    id_str = str(row.get("Unique_Pod_ID", ""))
    match = re.match(r'^([A-Za-z]+\d+)', id_str)
    if match:
        return match.group(1).upper()
    return "UNKNOWN"

audit_df["Season_Group"] = audit_df.apply(extract_season_prefix, axis=1)

# 3. Filter out pods 100 or higher based on our true numeric extraction
audit_df = audit_df[audit_df["True_Numeric_Pod"] < 100]

# 4. Sort FLUSH: Sort first by Season Group, then by the TRUE numerical pod order (1, 2, 3... 8, 9, 10... 80)
audit_df = audit_df.sort_values(by=["Season_Group", "True_Numeric_Pod"]).reset_index(drop=True)

print("================== 🔍 MASTER SEQUENTIAL GAP AUDIT (FIXED) ==================")
print(f"Scanning {len(audit_df)} rows sorted by pure numerical chronological sequence...\n")

skipped_records = []
previous_pod = None
previous_season = ""
previous_id = ""

# 5. Row-by-Row Sequence Scanner
for idx, row in audit_df.iterrows():
    current_pod = row["True_Numeric_Pod"]
    current_season = row["Season_Group"]
    current_id = str(row.get("Unique_Pod_ID", ""))
    
    # If we switch to a completely new season, reset our tracking baseline without flagging a gap
    if current_season != previous_season:
        previous_pod = current_pod
        previous_season = current_season
        previous_id = current_id
        continue
        
    if previous_pod is not None:
        # Now that it's sorted numerically, ANY jump greater than 1 means a pod was skipped!
        if current_pod > previous_pod + 1:
            missing_range = list(range(previous_pod + 1, current_pod))
            
            skipped_records.append({
                "Season": current_season,
                "Missing Pods": missing_range,
                "Context": f"Gap found in {current_season} between Pod {previous_pod} and Pod {current_pod}",
                "Prior ID": previous_id,
                "Next ID": current_id
            })
            
    previous_pod = current_pod
    previous_season = current_season
    previous_id = current_id

# 6. Final Report
if len(skipped_records) == 0:
    print("✅ SYSTEM HEALTHY: No unexpected sequential gaps found below Pod 100 across any season!")
else:
    print(f"⚠️ SYSTEMIC SEQUENTIAL GAPS DETECTED ({len(skipped_records)} true gaps found):")
    print("-" * 80)
    for gap in skipped_records:
        print(f"Season Sector: {gap['Season']}")
        print(f"Missing Pods:  {gap['Missing Pods']}")
        print(f"Details:       {gap['Context']}")
        print(f"Last Seen:     {gap['Prior ID']}")
        print(f"Resumed At:    {gap['Next ID']}")
        print("-" * 80)

================== 🔍 MASTER SEQUENTIAL GAP AUDIT (FIXED) ==================
Scanning 3973 rows sorted by pure numerical chronological sequence...

⚠️ SYSTEMIC SEQUENTIAL GAPS DETECTED (2 true gaps found):
--------------------------------------------------------------------------------
Season Sector: F24
Missing Pods:  [78]
Details:       Gap found in F24 between Pod 77 and Pod 79
Last Seen:     F24_Pod_77
Resumed At:    F24_Pod_79
--------------------------------------------------------------------------------
Season Sector: F25
Missing Pods:  [86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98]
Details:       Gap found in F25 between Pod 85 and Pod 99
Last Seen:     F25_Pod_85
Resumed At:    F25_Pod_99
--------------------------------------------------------------------------------


In [23]:
# 1. Isolate the exact Sp26 cohort and sort chronologically by index
sp26_raw = master_df[master_df["Unique_Pod_ID"].astype(str).str.contains("Sp26", case=False, na=False)]

# 2. Grab the exact row records around where 68 should sit
# We'll search for the literal text in the original string column before any parsing
print("================== 🔍 RAW UNIQUE_POD_ID STRING INVESTIGATION ==================")
string_matches = master_df[master_df["Unique_Pod_ID"].astype(str).str.contains("68", na=False)]

if len(string_matches) > 0:
    print(f"Found {len(string_matches)} raw string match(es) for '68':")
    for idx, row in string_matches.iterrows():
        raw_id = row["Unique_Pod_ID"]
        print(f"DataFrame Index: {idx} | Raw Value: '{raw_id}' | Length: {len(str(raw_id))}")
else:
    print("❌ No text strings containing '68' exist in the current DataFrame.")
    print("This confirms a physical data ingestion cutoff. Let's inspect the boundaries.")

================== 🔍 RAW UNIQUE_POD_ID STRING INVESTIGATION ==================
Found 26 raw string match(es) for '68':
DataFrame Index: 1364 | Raw Value: 'F24_Pod_68' | Length: 10
DataFrame Index: 1365 | Raw Value: 'F24_Pod_68' | Length: 10
DataFrame Index: 1366 | Raw Value: 'F24_Pod_68' | Length: 10
DataFrame Index: 1367 | Raw Value: 'F24_Pod_68' | Length: 10
DataFrame Index: 1368 | Raw Value: 'F24_Pod_68' | Length: 10
DataFrame Index: 1369 | Raw Value: 'F24_Pod_68' | Length: 10
DataFrame Index: 1370 | Raw Value: 'F24_Pod_68' | Length: 10
DataFrame Index: 1371 | Raw Value: 'F24_Pod_68' | Length: 10
DataFrame Index: 1372 | Raw Value: 'F24_Pod_68' | Length: 10
DataFrame Index: 2114 | Raw Value: 'F25_Pod_68' | Length: 10
DataFrame Index: 2115 | Raw Value: 'F25_Pod_68' | Length: 10
DataFrame Index: 2116 | Raw Value: 'F25_Pod_68' | Length: 10
DataFrame Index: 2117 | Raw Value: 'F25_Pod_68' | Length: 10
DataFrame Index: 2118 | Raw Value: 'F25_Pod_68' | Length: 10
DataFrame Index: 2119 | Raw

In [24]:
# 1. Isolate the exact Sp26 cohort
sp26_raw = master_df[master_df["Unique_Pod_ID"].astype(str).str.contains("Sp26", case=False, na=False)]

# 2. Sort it chronologically by the true data frame index to see how it was read row-by-row
sp26_ordered = sp26_raw.sort_index()

print("================== 🔍 SP26 NEIGHBORHOOD SEQUENCE PROBE ==================")

# 3. Find the index boundary entries for Pod 67 and Pod 69
# This lets us find exactly where Pod 68 should be wedged in
pod_67_indices = sp26_ordered[sp26_ordered["Pod"] == 67].index
pod_69_indices = sp26_ordered[sp26_ordered["Pod"] == 69].index

if len(pod_67_indices) > 0 and len(pod_69_indices) > 0:
    max_67_idx = max(pod_67_indices)
    min_69_idx = min(pod_69_indices)
    
    print(f"Pod 67 entries end at DataFrame Index: {max_67_idx}")
    print(f"Pod 69 entries begin at DataFrame Index: {min_69_idx}")
    print(f"Gaps in indices between them: {min_69_idx - max_67_idx - 1} hidden rows.\n")
    
    # 4. Print the exact sequence wrap right around the gap
    print("--- Visual Sequence Wrap ---")
    boundary_display = master_df.loc[max_67_idx-2 : min_69_idx+2]
    print(boundary_display[["Unique_Pod_ID", "Pod", "Name_Clean", "Pick 1", "Pick 2"]])

else:
    print("Could not isolate consecutive boundaries. Let's look at the end of the Sp26 dataset:")
    print(sp26_ordered[["Unique_Pod_ID", "Pod", "Name_Clean"]].tail(10))
    

================== 🔍 SP26 NEIGHBORHOOD SEQUENCE PROBE ==================
Pod 67 entries end at DataFrame Index: 4071
Pod 69 entries begin at DataFrame Index: 4080
Gaps in indices between them: 8 hidden rows.

--- Visual Sequence Wrap ---
     Unique_Pod_ID  Pod                              Name_Clean  \
4069   Sp26_Pod_67   67                    Forever A Team & Son   
4070   Sp26_Pod_67   67                             energythief   
4071   Sp26_Pod_67   67                                    Jalt   
4072   Sp26_Pod_68   68                               Doormatte   
4073   Sp26_Pod_68   68                               Freeclint   
4074   Sp26_Pod_68   68  SharkNate-O & onthegeniusofsamrockwell   
4075   Sp26_Pod_68   68                       NadRao & Sparkles   
4076   Sp26_Pod_68   68                             Jmaster1324   
4077   Sp26_Pod_68   68            thedoctormate0806 & mrrj6549   
4078   Sp26_Pod_68   68                       Astigmatic_Oracle   
4079   Sp26_Pod_68   68  

In [25]:
import pandas as pd

# 1. Load the RAW, uncleaned CSV file directly before your cleaner script touches it
# (Replace with your actual raw file path)
raw_file_path = "dirty results/Sp26W6.csv" 

try:
    raw_df = pd.read_csv(raw_file_path)
    print(f"=== RAW FILE INSPECTION: {raw_file_path} ===")
    print(f"Total raw rows found in file: {len(raw_df)}")
    
    # 2. Search for '68' anywhere in the raw text of the entire file
    # We convert the whole row to a string to find where it's hiding
    raw_matches = raw_df[raw_df.astype(str).apply(lambda x: x.str.contains('68')).any(axis=1)]
    
    if len(raw_matches) > 0:
        print(f"\n✅ SUCCESS: Found {len(raw_matches)} rows containing '68' in the raw file!")
        print("Here is exactly what they look like before cleaning:")
        print(raw_matches.head(10).to_string())
    else:
        print("\n❌ CRITICAL: '68' is completely missing from the raw CSV file itself.")
        print("This means the issue happened during the Google Sheets download/export step.")
        
except Exception as e:
    print(f"Error reading raw file: {e}")

=== RAW FILE INSPECTION: dirty results/Sp26W6.csv ===
Total raw rows found in file: 167

✅ SUCCESS: Found 9 rows containing '68' in the raw file!
Here is exactly what they look like before cleaning:
      Pod                       Setup  Rank Points Earned  Total Points                     Name(s) Victory\nPile 1 Victory\nPile 2 Total\nVP Win? Unnamed: 10 Turns Unnamed: 12 Escaped Villains Unnamed: 14 Points per Turn Unnamed: 16 Total Rank Points Unnamed: 18 VP Difference Bystanders Rescued Cards Undercover Cards Left in Villain Deck Unnamed: 23 Unnamed: 24 Unnamed: 25 Unnamed: 26 Unnamed: 27 Unnamed: 28   Attempt 1 Hero Choices             Unnamed: 30            Attempt 2 Hero Choices                       Unnamed: 32 Attempt Submitted Fun Level                                                                                                                                                                                                                                                    

In [26]:
import pandas as pd
import numpy as np

# Load the raw file
test_df = pd.read_csv("dirty results/Sp26W6.csv")

print("=== CLEANER EXECUTION SIMULATION ON POD 68 ===")

# Let's recreate your cleaner's forward fill step
test_df["Pod"] = test_df["Pod"].ffill()

# Check if Pod 68 survived the initial load/fill
pod_68_sim = test_df[test_df["Pod"] == 68]
print(f"Rows assigned to Pod 68 after forward-fill step: {len(pod_68_sim)}")

# Let's test the common filtering blocks your cleaner likely uses:
# Test 1: Dropping rows missing critical data
missing_hero = pod_68_sim[pod_68_sim["Attempt 1 Hero Choices"].isna()]
print(f" -> Rows dropped due to missing Hero selections: {len(missing_hero)}")

# Test 2: Text parsing verification
# Does your cleaner check if the player name is valid or if strings are clean?
for idx, row in pod_68_sim.iterrows():
    player = row.get("Name(s)", "Unknown")
    hero1 = row.get("Attempt 1 Hero Choices", "Unknown")
    
    # Simulate a typical cleaning validation check:
    if pd.isna(hero1) or str(hero1).strip() == "":
        print(f"   ❌ Row {idx} ({player}) would be dropped: Hero column is blank.")
    elif pd.isna(player) or str(player).strip() == "":
        print(f"   ❌ Row {idx} would be dropped: Player Name column is blank.")
    else:
        print(f"   ✅ Row {idx} ({player} playing {hero1}) passes basic validation.")

=== CLEANER EXECUTION SIMULATION ON POD 68 ===
Rows assigned to Pod 68 after forward-fill step: 11
 -> Rows dropped due to missing Hero selections: 2
   ✅ Row 101 (Doormatte playing Bruce Banner) passes basic validation.
   ✅ Row 102 (Freeclint playing Squirrel Girl) passes basic validation.
   ✅ Row 103 (SharkNate-O & onthegeniusofsamrockwell playing Squirrel Girl) passes basic validation.
   ✅ Row 104 (NadRao & Sparkles playing Bruce Banner) passes basic validation.
   ✅ Row 105 (Jmaster1324 playing Bruce Banner) passes basic validation.
   ✅ Row 106 (thedoctormate0806 & mrrj6549 playing Bruce Banner) passes basic validation.
   ✅ Row 107 (Astigmatic_Oracle playing Invisible Woman) passes basic validation.
   ✅ Row 108 (Lasombra81 playing Bruce Banner) passes basic validation.
   ✅ Row 109 (MysticRuby playing -) passes basic validation.
   ❌ Row 110 (nan) would be dropped: Hero column is blank.
   ❌ Row 111 (nan) would be dropped: Hero column is blank.


In [27]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4375 entries, 0 to 4374
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Pod                4375 non-null   int32  
 1   Rank               4375 non-null   object 
 2   Name_Clean         4375 non-null   object 
 3   VP_Pile_1          4375 non-null   int32  
 4   VP_Pile_2          4375 non-null   int32  
 5   Total_VP           4375 non-null   int32  
 6   Win?               4375 non-null   object 
 7   Total Rank Points  4375 non-null   object 
 8   VP_Diff            4375 non-null   int32  
 9   Bystanders         4375 non-null   object 
 10  Fun_Level          4375 non-null   object 
 11  Win_Value          4375 non-null   object 
 12  Win_Points         4375 non-null   float64
 13  Turns_Value        4375 non-null   object 
 14  Turns_Points       4375 non-null   object 
 15  Turns_Rank         4375 non-null   object 
 16  Escapes_Value      4375 

In [28]:
# ==========================================
# STEP 1: LOAD THE RAW FILE & SIMULATE THE FFILL
# ==========================================
file_test = "dirty results/Sp26W6.csv"   # <-- Put one of your problem file paths here
df_raw = pd.read_csv(file_test, encoding='utf-8-sig')
df_raw.columns = df_raw.columns.str.strip()

# Set target pod to observe
target_pod = 68                          # <-- Put your target pod number here

# Run your exact forward fill to catch the rows assigned to that pod
df_raw['Pod'] = df_raw['Pod'].replace(['-', '0', '0.0', 'nan', 'None'], np.nan).ffill()
df_raw['Pod'] = pd.to_numeric(df_raw['Pod'], errors='coerce')

# Isolate the exact chunk of rows that belong to your pod *before* cleaning
pod_chunk_raw = df_raw[df_raw['Pod'] == target_pod]

print(f"=== DIAGNOSTIC FOR POD {target_pod} ===")
print(f"Total raw rows captured in this pod section: {len(pod_chunk_raw)}\n")
print("--- RAW DATA COPIED FOR THIS POD ---")
print(pod_chunk_raw[['Name', 'Pod']].to_string() if 'Name' in pod_chunk_raw.columns else pod_chunk_raw)


# ==========================================
# STEP 2: RUN INDIVIDUAL FILTERS TO SEE WHO DIES
# ==========================================
print("\n=== STEP-BY-STEP PIPELINE VACCINATION TEST ===")

# Test A: The Header / Name Clean Phase
df_step_a = dynamically_rename_columns(df_raw.copy(), official_names)
pod_chunk_a = df_step_a[df_step_a['Pod'] == target_pod]
print(f"Rows remaining after Column Renaming/Mapping: {len(pod_chunk_a)}")

# Test B: The Anchor of Truth Row Purge
if 'Win_Value' in df_step_a.columns:
    df_step_b = df_step_a[
        df_step_a['Win_Value'].notna() & 
        ~df_step_a['Win_Value'].astype(str).str.lower().isin(['nan', 'none', '-', '']) &
        df_step_a['Win_Points'].notna()
    ]
    pod_chunk_b = df_step_b[df_step_b['Pod'] == target_pod]
    print(f"Rows remaining after Anchor Row Purge (Missing Scores filter): {len(pod_chunk_b)}")
else:
    df_step_b = df_step_a
    print("Skipped Anchor Row Purge: 'Win_Value' column was NOT generated for this file!")

# Test C: The Hero Pick Lookup Drop
df_step_c = hero_picks(df_step_b.copy(), official_names)
pod_chunk_c = df_step_c[df_step_c['Pod'] == target_pod]
print(f"Rows remaining after Hero Pick Verification Drop: {len(pod_chunk_c)}")

=== DIAGNOSTIC FOR POD 68 ===
Total raw rows captured in this pod section: 11

--- RAW DATA COPIED FOR THIS POD ---
      Pod            Setup  Rank Points Earned  Total Points  \
101  68.0  Gadget Gumshoes   1.0          (+9)          26.0   
102  68.0              NaN   2.0          (+8)          26.0   
103  68.0              NaN   3.0          (+7)          30.0   
104  68.0              NaN   4.0          (+6)          28.0   
105  68.0              NaN   5.0          (+5)          24.0   
106  68.0              NaN   6.0          (+4)          23.0   
107  68.0              NaN   7.0          (+3)          26.0   
108  68.0              NaN   8.0          (+2)          15.0   
109  68.0              NaN   9.0          (+1)           3.0   
110  68.0              NaN  10.0           NaN           NaN   
111  68.0              NaN  11.0           NaN           NaN   

                                    Name(s) Victory\nPile 1 Victory\nPile 2  \
101                               Do

KeyError: 'Unique_Pod_ID'

In [30]:
all_picks = set(master_df['Pick 1'].dropna().unique()).union(set(master_df['Pick 2'].dropna().unique()))
invalid_picks = all_picks - official_names

print(f"Non-conforming heroes found in master_df: {invalid_picks if invalid_picks else 'None! All clean.'}")

Non-conforming heroes found in master_df: None! All clean.
